In [ ]:
"""
Analyze per-pair min/max statistics of the real LR depth (phone ToF,
LR_fill_depth.png) vs. the HR GT depth (HR_gt.png) for the RGBDD-Full
real-world dataset, and quantify how often the HR GT falls outside the
LR min/max range (i.e., pixels that get clipped when the HR target is
normalized with LR statistics, as in ProcessingRGBDDReal).

Clipping ranges mirror the pipeline and are category-specific:
    models:    [0.6, 3.0] m
    portraits: [1.0, 5.0] m
    plants:    [0.6, 1.5] m

Usage:
    python analyze_rgbdd_minmax.py --base /path/to/RGBDD-Full --split train
    python analyze_rgbdd_minmax.py --base /path/to/RGBDD-Full --split test

Outputs a CSV (one row per pair) and prints a summary, overall and
per category.
"""

import argparse
import csv
import os

import numpy as np
from PIL import Image

# (category, subfolder pattern, low, high) — mirrors GenerateNPYFiles
CATEGORIES = [
    ("models", "models/models_{split}", 0.6, 3.0),
    ("portraits", "portraits/portraits_{split}", 1.0, 5.0),
    ("plants", "plants/plants_{split}", 0.6, 1.5),
]


def load_pair_paths(path: str):
    """Mirror ProcessingRGBDDReal._LoadPairPaths: walk and collect
    (hr_gt, rgb, lr) triples per directory."""
    pairs = []
    for root, _, files in os.walk(path):
        depth_hr = depth_lr = rgb = None
        for file in files:
            if file.endswith("HR_gt.png"):
                depth_hr = os.path.join(root, file)
            if file.endswith("LR_fill_depth.png"):
                depth_lr = os.path.join(root, file)
            if file.endswith("RGB.jpg"):
                rgb = os.path.join(root, file)
        if rgb is None or depth_hr is None or depth_lr is None:
            continue  # incomplete folder (the pipeline just prints these)
        pairs.append((depth_hr, rgb, depth_lr))
    return pairs


def load_depth_m(path: str) -> np.ndarray:
    """Load a depth PNG stored in millimeters, return float32 meters."""
    return np.asarray(Image.open(path), dtype=np.float32) / 1000.0


def analyze_pair(hr_path: str, lr_path: str, low: float, high: float):
    hr = np.clip(load_depth_m(hr_path), low, high)
    lr = np.clip(load_depth_m(lr_path), low, high)

    hr_min, hr_max = float(hr.min()), float(hr.max())
    lr_min, lr_max = float(lr.min()), float(lr.max())

    n_pix = hr.size
    return {
        "lr_min": lr_min,
        "lr_max": lr_max,
        "hr_min": hr_min,
        "hr_max": hr_max,
        "min_diff_hr_minus_lr": hr_min - lr_min,
        "max_diff_hr_minus_lr": hr_max - lr_max,
        "lr_range": lr_max - lr_min,
        "hr_range": hr_max - hr_min,
        # HR pixels that _NormalizeDepthWithMinMax would clip to 0/1
        "frac_hr_above_lr_max": float((hr > lr_max).sum()) / n_pix,
        "frac_hr_below_lr_min": float((hr < lr_min).sum()) / n_pix,
        # Worst-case error (in meters) introduced by that clipping
        "clip_overshoot_m": max(hr_max - lr_max, 0.0),
        "clip_undershoot_m": max(lr_min - hr_min, 0.0),
    }


def summarize(rows, label):
    def col(name):
        return np.array([r[name] for r in rows], dtype=np.float64)

    if not rows:
        print(f"\n===== {label}: no pairs =====")
        return

    max_diff = col("max_diff_hr_minus_lr")
    min_diff = col("min_diff_hr_minus_lr")
    frac_above = col("frac_hr_above_lr_max")
    frac_below = col("frac_hr_below_lr_min")
    over = col("clip_overshoot_m")
    under = col("clip_undershoot_m")

    print(f"\n===== {label} ({len(rows)} pairs) =====")
    print(f"max_diff (hr_max - lr_max):  mean {max_diff.mean():+.4f} m | "
          f"median {np.median(max_diff):+.4f} | min {max_diff.min():+.4f} | "
          f"max {max_diff.max():+.4f}")
    print(f"min_diff (hr_min - lr_min):  mean {min_diff.mean():+.4f} m | "
          f"median {np.median(min_diff):+.4f} | min {min_diff.min():+.4f} | "
          f"max {min_diff.max():+.4f}")
    print(f"pairs where HR exceeds LR max: {(max_diff > 0).sum()} "
          f"({100.0 * (max_diff > 0).mean():.1f}%)")
    print(f"pairs where HR goes below LR min: {(min_diff < 0).sum()} "
          f"({100.0 * (min_diff < 0).mean():.1f}%)")
    print(f"HR pixels clipped above (per pair): mean {100 * frac_above.mean():.3f}% | "
          f"worst {100 * frac_above.max():.3f}%")
    print(f"HR pixels clipped below (per pair): mean {100 * frac_below.mean():.3f}% | "
          f"worst {100 * frac_below.max():.3f}%")
    print(f"worst-case clipping error: overshoot {over.max():.4f} m, "
          f"undershoot {under.max():.4f} m")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", required=True,
                    help="Path to the RGBDD-Full folder")
    ap.add_argument("--split", choices=["train", "test"], default="train")
    ap.add_argument("--out", default=None, help="Output CSV path")
    args = ap.parse_args()

    out_csv = args.out or f"rgbdd_minmax_{args.split}.csv"

    all_rows = []
    for category, pattern, low, high in CATEGORIES:
        folder = os.path.join(args.base, pattern.format(split=args.split))
        pairs = load_pair_paths(folder)
        print(f"{category}: {len(pairs)} pairs (clip [{low}, {high}] m)")

        for idx, (hr_path, _rgb_path, lr_path) in enumerate(pairs):
            row = analyze_pair(hr_path, lr_path, low, high)
            row_meta = {
                "category": category,
                "index": idx,
                "hr_path": os.path.relpath(hr_path, args.base),
                "clip_low": low,
                "clip_high": high,
            }
            all_rows.append({**row_meta, **row})

            if (idx + 1) % 100 == 0:
                print(f"  {category}: processed {idx + 1}/{len(pairs)}")

    if not all_rows:
        print("No pairs found — check --base path.")
        return

    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
        writer.writeheader()
        writer.writerows(all_rows)

    # Per-category summaries, then overall
    for category, _, low, high in CATEGORIES:
        cat_rows = [r for r in all_rows if r["category"] == category]
        summarize(cat_rows, f"{category} [{low}, {high}] m ({args.split})")
    summarize(all_rows, f"ALL categories ({args.split})")

    print(f"\nPer-pair results written to {out_csv}")


In [ ]:
main()